In [ ]:
import pandas as pd
import numpy as np

# safe filler codes from real ukb data
SAFE_MED_CODES = [1140875408, 1140879802, 1140879778, 1140916356, 1140861958]
# these will be filtered out by the pipeline
EXCLUDED_CODES = [1140884600, 1140871310, 1140874686, 1140883066]

def generate_ukb_full(n_samples=50, cohort_type='healthy'):
    """
    Generate synthetic ukb-format data that passes the preprocessing script.
    
    compatibility requirements:
    - no nan in any column to avoid aggressive dropna
    - stress fields in range 1 to 4
    - glucose in raw ukb units
    - no excluded medication codes to pass the medication wall
    """
    np.random.seed(42 if cohort_type == 'healthy' else 24)
    data = {}

    # identifiers and demographics
    start_eid = 1000 if cohort_type == 'healthy' else 2000
    data['participant.eid'] = range(start_eid, start_eid + n_samples)
    data['participant.p31'] = np.random.choice([0, 1], n_samples)       # sex
    data['participant.p21022'] = np.random.randint(45, 75, n_samples)   # age

    # anthropometry including height, weight, and bmi
    heights = np.random.normal(172, 9, n_samples)
    if cohort_type == 'healthy':
        bmis = np.random.uniform(18.5, 24.9, n_samples)
    else:
        bmis = np.random.uniform(30.0, 42.0, n_samples)

    weights = bmis * (heights / 100) ** 2
    data['participant.p50_i0'] = np.round(heights, 1)       # height
    data['participant.p21002_i0'] = np.round(weights, 1)    # weight
    data['participant.p21001_i0'] = np.round(bmis, 1)       # bmi

    # metabolic markers using raw ukb glucose units
    # preprocessing script will multiply by 18.0182
    if cohort_type == 'healthy':
        glucose_mmol = np.random.uniform(3.9, 5.4, n_samples)
        hba1c = np.random.uniform(28.0, 41.0, n_samples)
        crp = np.round(np.random.lognormal(mean=0, sigma=0.5, size=n_samples) * 0.5, 2)
        igf1 = np.round(np.random.normal(22, 3, n_samples), 2)
    else:
        glucose_mmol = np.random.uniform(7.2, 16.0, n_samples)
        hba1c = np.random.uniform(48.0, 90.0, n_samples)
        crp = np.round(np.random.lognormal(mean=1.5, sigma=0.6, size=n_samples), 2)
        igf1 = np.round(np.random.normal(14, 3, n_samples), 2)

    data['participant.p30740_i0'] = np.round(glucose_mmol, 2)  # glucose
    data['participant.p30750_i0'] = np.round(hba1c, 1)         # hba1c
    data['participant.p30710_i0'] = crp                        # crp
    data['participant.p30770_i0'] = igf1                       # igf-1

    # blood pressure readings
    if cohort_type == 'healthy':
        sys_bp = np.random.randint(110, 128, n_samples)
        dia_bp = np.random.randint(70, 84, n_samples)
    else:
        sys_bp = np.random.randint(135, 165, n_samples)
        dia_bp = np.random.randint(85, 105, n_samples)

    data['participant.p4080_i0_a0'] = sys_bp
    data['participant.p4080_i0_a1'] = sys_bp + np.random.randint(-2, 3, n_samples)
    data['participant.p4079_i0_a0'] = dia_bp
    data['participant.p4079_i0_a1'] = dia_bp + np.random.randint(-2, 3, n_samples)

    # medication must not be nan otherwise the pipeline drops the row
    # use safe filler codes that are not metformin or insulin
    for i in range(8):
        col = f'participant.p20003_i0_a{i}'
        data[col] = np.random.choice(SAFE_MED_CODES, n_samples)

    # mental health symptoms items using raw ukb scale 1-4
    mh_fields = [
        'p20510', 'p20507', 'p20519', 'p20514', 'p20511', 'p20513',
        'p20508', 'p20518', 'p20505', 'p20512', 'p20506', 'p20509',
        'p20516', 'p20515', 'p20520', 'p20517'
    ]
    for field in mh_fields:
        if cohort_type == 'healthy':
            data[f'participant.{field}'] = np.random.choice([1, 2], n_samples, p=[0.80, 0.20])
        else:
            data[f'participant.{field}'] = np.random.choice([1, 2, 3, 4], n_samples, p=[0.25, 0.35, 0.25, 0.15])

    df = pd.DataFrame(data)

    # final checks to ensure everything is within range
    assert df.isnull().sum().sum() == 0, "nan found which will break the pipeline"
    g_mgdl = df['participant.p30740_i0'] * 18.0182
    if cohort_type == 'healthy':
        assert (g_mgdl >= 70).all() and (g_mgdl < 100).all(), "healthy glucose range failure"
    else:
        assert (g_mgdl >= 126).all(), "diabetic glucose below threshold"

    return df

# generate and save the pilot cohorts
df_healthy = generate_ukb_full(50, 'healthy')
df_diabetic = generate_ukb_full(50, 'diabetic')

df_healthy.to_csv('t2d_pilot_healthy_full.csv', index=False)
df_diabetic.to_csv('t2d_diabetics_full.csv', index=False)

# output counts to verify the pipeline will see 50 per group
print(f"Healthy: {len(df_healthy)} rows")
print(f"Diabetic: {len(df_diabetic)} rows")

Healthy: 50 rows
Diabetic: 50 rows
